# **Basic Programming in Python**

## Summer Term 2025

### Application Tutorial: Reinforcement Learning with a Two-Armed Bandit

## **Part 1: Introduction to Reinforcement Learning**

### What is Reinforcement Learning?

**Reinforcement Learning (RL)** is a type of learning where an **agent** learns by interacting with an **environment**. The agent takes actions, and the environment gives back **rewards** (or punishments). Over time, the agent learns which actions lead to the most reward.

This is different from learning with a textbook (where someone tells you the right answer). In RL, the agent has to **discover** good strategies through trial and error -- just like how we learn many things in real life!

### The Exploration vs. Exploitation Dilemma

One of the central challenges in reinforcement learning is the **exploration vs. exploitation dilemma**:

- **Exploitation**: Stick with what you already know works well. Choose the action that has given you the highest reward so far.
- **Exploration**: Try something new. Maybe there is an even better action you haven't discovered yet!

The challenge is finding the right **balance**. If you only exploit, you might miss out on better options. If you only explore, you waste time on bad actions even when you already know a good one.

**Real-world examples:**
- A **child learning which foods they like**: Do you eat your favorite food every meal (exploit), or try something new that might be even better (explore)?
- A **rat pressing levers** in a Skinner box: One lever gives food pellets more often than the other. The rat has to figure out which one is better, while still occasionally trying the other lever to make sure.

### The Two-Armed Bandit Problem

The **two-armed bandit** is the simplest reinforcement learning problem. It is named after slot machines (sometimes called "one-armed bandits").

Imagine you are in front of **two slot machines**. Each machine pays out a reward with some **unknown probability**. Your goal is to **maximize your total reward** over many plays.

- You don't know which machine is better.
- Each time you play, you choose one machine and observe whether you win (reward = 1) or lose (reward = 0).
- Over time, you need to figure out which machine is better and play it more often.

In this tutorial, we will build a two-armed bandit environment and then create a **Q-learning agent** that learns to choose the better arm!

---

## **Part 2: Building the Bandit Environment**

Let's start by building the bandit environment step by step using functions and NumPy.

### Exercise 2.1: Create the Bandit

First, we need a function that creates our two-armed bandit. The bandit is simply defined by two reward probabilities -- one for each arm.

In [ ]:
import numpy as np

def create_bandit(p_arm_0, p_arm_1):
    """Create a two-armed bandit with given reward probabilities.
    
    Args:
        p_arm_0 (float): Probability of reward for arm 0 (between 0 and 1).
        p_arm_1 (float): Probability of reward for arm 1 (between 0 and 1).
    
    Returns:
        list: A list containing the two reward probabilities.
    """
    return [p_arm_0, p_arm_1]

# Create a bandit where arm 0 pays out 30% of the time and arm 1 pays out 70%
bandit = create_bandit(0.3, 0.7)
print(f"Bandit reward probabilities: {bandit}")

### Exercise 2.2: Pull an Arm

Now we need a function that simulates pulling one arm of the bandit. When we pull an arm, we get a reward of **1** with the arm's probability, and **0** otherwise.

**Your task:** Implement the `pull_arm` function.

*Hint: Use `np.random.random()` which returns a random float between 0 and 1. Compare it to the arm's probability. If the random number is less than the probability, the arm pays out!*

In [ ]:
def pull_arm(bandit, arm):
    """Pull one arm of the bandit and receive a reward.
    
    Args:
        bandit (list): The reward probabilities for each arm.
        arm (int): Which arm to pull (0 or 1).
    
    Returns:
        int: 1 if rewarded, 0 if not.
    """
    pass

# Test: pull arm 1 ten times and see what happens
for i in range(10):
    reward = pull_arm(bandit, 1)
    print(f"Pull {i+1}: Reward = {reward}")

### Exercise 2.3: Run a Simple Experiment

Before we build a smart agent, let's see how well a **random strategy** performs. In each trial, we randomly choose an arm and record the result.

**Your task:** Implement the `run_random_experiment` function.

*Hint: Use `np.random.choice([0, 1])` to randomly pick an arm each trial.*

In [ ]:
def run_random_experiment(bandit, n_trials):
    """Run an experiment where a random arm is chosen each trial.
    
    Args:
        bandit (list): The reward probabilities.
        n_trials (int): Number of trials.
    
    Returns:
        tuple: (choices, rewards) - lists of chosen arms and received rewards.
    """
    pass

choices, rewards = run_random_experiment(bandit, 1000)
print(f"Total reward with random strategy: {sum(rewards)}")
print(f"Average reward per trial: {np.mean(rewards):.3f}")

---

## **Part 3: The Q-Learning Agent**

Now let's build a smarter agent that actually **learns** from experience!

### Q-Values: What the Agent Believes

The agent maintains a set of **Q-values** -- one for each arm. A Q-value represents the agent's current **estimate** of how good an action is (i.e., how much reward it expects to get from choosing that arm).

For our two-armed bandit, the Q-table is simply:

```
Q = [Q_0, Q_1]
```

We initialize both Q-values to **0.0** (the agent starts with no knowledge).

### The Update Rule: Learning from Experience

After each trial, the agent updates its Q-value for the chosen arm using this rule:

```
Q[arm] = Q[arm] + alpha * (reward - Q[arm])
```

Let's break this down:

- **`alpha`** (learning rate, between 0 and 1): How much the agent adjusts its belief after each trial. A high alpha means the agent changes its mind quickly; a low alpha means it changes slowly.
- **`reward - Q[arm]`**: This is the **prediction error** -- the difference between what actually happened and what the agent expected. If the reward was higher than expected, the prediction error is positive. If lower, it's negative.

The agent **nudges its Q-value in the direction of the prediction error**, scaled by the learning rate.

### Prediction Error and Dopamine

The prediction error `(reward - Q[arm])` is not just a mathematical trick -- it has a deep connection to neuroscience! Dopamine neurons in the brain behave in a remarkably similar way. We will return to this connection at the end of the tutorial.

### The Epsilon-Greedy Strategy: Balancing Exploration and Exploitation

How does the agent decide which arm to pull? We use the **epsilon-greedy** strategy:

- With probability **epsilon** (`epsilon`): **explore** -- choose a random arm.
- With probability **1 - epsilon**: **exploit** -- choose the arm with the highest Q-value.

For example, with `epsilon = 0.1`, the agent explores randomly 10% of the time and exploits its best-known option 90% of the time.

### Exercise 3.1: Initialize Q-Values

This one is simple -- we just need both Q-values to start at zero.

In [ ]:
def initialize_q_values():
    """Initialize Q-values for both arms to zero.
    
    Returns:
        list: Q-values [Q_0, Q_1] both set to 0.0.
    """
    return [0.0, 0.0]

### Exercise 3.2: Choose an Action

**Your task:** Implement the `choose_action` function using the epsilon-greedy strategy described above.

*Hint: Generate a random number with `np.random.random()`. If it is less than epsilon, choose a random arm with `np.random.choice([0, 1])`. Otherwise, choose the arm with the higher Q-value. You can use `np.argmax()` for that!*

In [ ]:
def choose_action(q_values, epsilon):
    """Choose an action using epsilon-greedy strategy.
    
    Args:
        q_values (list): Current Q-values for each arm.
        epsilon (float): Exploration rate (0 to 1).
    
    Returns:
        int: The chosen arm (0 or 1).
    """
    pass

### Exercise 3.3: Update Q-Values

**Your task:** Implement the `update_q_value` function using the update rule from above:

```
Q[arm] = Q[arm] + alpha * (reward - Q[arm])
```

*Remember to return the updated Q-values list!*

In [ ]:
def update_q_value(q_values, arm, reward, alpha):
    """Update the Q-value for the chosen arm.
    
    Args:
        q_values (list): Current Q-values.
        arm (int): The arm that was chosen.
        reward (int): The reward received (0 or 1).
        alpha (float): Learning rate (0 to 1).
    
    Returns:
        list: Updated Q-values.
    """
    pass

---

## **Part 4: Running the Q-Learning Agent**

Now let's put all the pieces together and run our Q-learning agent on the bandit!

### Exercise 4.1: The Full Q-Learning Loop

**Your task:** Fill in the loop body. In each trial, the agent should:
1. Choose an action (using `choose_action`)
2. Pull the arm and get a reward (using `pull_arm`)
3. Update Q-values (using `update_q_value`)
4. Store the results in the provided lists

In [ ]:
def run_q_learning(bandit, n_trials, alpha, epsilon):
    """Run a Q-learning agent on the two-armed bandit.
    
    Args:
        bandit (list): Reward probabilities.
        n_trials (int): Number of trials.
        alpha (float): Learning rate.
        epsilon (float): Exploration rate.
    
    Returns:
        tuple: (q_values, choices, rewards, q_history)
    """
    q_values = initialize_q_values()
    choices = []
    rewards = []
    q_history = []  # store q_values at each step for plotting
    
    for trial in range(n_trials):
        # 1. Choose an action
        # 2. Pull the arm and get reward
        # 3. Update Q-values
        # 4. Store results
        pass
    
    return q_values, choices, rewards, q_history

### Exercise 4.2: Run and Evaluate

Now let's run the agent and see how it performs! Run the cell below (make sure you have implemented all the functions above first).

In [ ]:
# Run the agent
bandit = create_bandit(0.3, 0.7)
q_values, choices, rewards, q_history = run_q_learning(bandit, 1000, alpha=0.1, epsilon=0.1)

print(f"Final Q-values: {q_values}")
print(f"Total reward: {sum(rewards)}")
print(f"Average reward: {np.mean(rewards):.3f}")
print(f"Arm 0 chosen: {choices.count(0)} times")
print(f"Arm 1 chosen: {choices.count(1)} times")

---

## **Part 5: Visualizing Learning**

Let's create some plots to understand how our agent learns over time. Run the cell below to generate three plots.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Q-values over time ---
q_history_array = np.array(q_history)
axes[0].plot(q_history_array[:, 0], label="Q(arm 0)", alpha=0.8)
axes[0].plot(q_history_array[:, 1], label="Q(arm 1)", alpha=0.8)
axes[0].axhline(y=bandit[0], color="blue", linestyle="--", alpha=0.5, label=f"True p(arm 0) = {bandit[0]}")
axes[0].axhline(y=bandit[1], color="orange", linestyle="--", alpha=0.5, label=f"True p(arm 1) = {bandit[1]}")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Q-value")
axes[0].set_title("Q-Values Over Time")
axes[0].legend()
axes[0].set_ylim(-0.1, 1.1)

# --- Plot 2: Cumulative reward ---
cumulative_rewards = np.cumsum(rewards)
trials = np.arange(1, len(rewards) + 1)
axes[1].plot(trials, cumulative_rewards, label="Q-learning agent", color="green")
axes[1].plot(trials, trials * max(bandit), "--", label=f"Best possible (always arm {np.argmax(bandit)})", color="gold")
axes[1].plot(trials, trials * np.mean(bandit), "--", label="Random strategy", color="gray")
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Cumulative Reward")
axes[1].set_title("Cumulative Reward Over Time")
axes[1].legend()

# --- Plot 3: Rolling average reward ---
window = 50
rolling_avg = np.convolve(rewards, np.ones(window)/window, mode="valid")
axes[2].plot(rolling_avg, color="purple", alpha=0.8)
axes[2].axhline(y=max(bandit), color="gold", linestyle="--", alpha=0.5, label=f"Best arm probability = {max(bandit)}")
axes[2].axhline(y=np.mean(bandit), color="gray", linestyle="--", alpha=0.5, label=f"Random expected = {np.mean(bandit)}")
axes[2].set_xlabel("Trial")
axes[2].set_ylabel(f"Average Reward (window={window})")
axes[2].set_title("Rolling Average Reward")
axes[2].legend()

plt.tight_layout()
plt.show()

---

## **Challenge: Tune Your Agent!**

Now it's time to experiment! Your goal is to maximize the total reward the agent collects over 1000 trials.

You have two knobs to turn:
- **`alpha` (learning rate)**: How quickly does the agent update its beliefs? (0.0 to 1.0)
- **`epsilon` (exploration rate)**: How often does the agent explore randomly? (0.0 to 1.0)

Think about these questions as you experiment:
- What happens with a very high learning rate? A very low one?
- What if epsilon is 0 (pure exploitation)? What if it's 1 (pure exploration)?
- Can you find the sweet spot?

Try at least 5 different parameter combinations and record your results below!

In [ ]:
# Parameter tweaking challenge!
# Try different values of alpha and epsilon
# Record your results

bandit = create_bandit(0.3, 0.7)

# Combination 1
alpha_1, epsilon_1 = 0.1, 0.1  # change these!
_, _, rewards_1, _ = run_q_learning(bandit, 1000, alpha_1, epsilon_1)

# Combination 2
alpha_2, epsilon_2 = 0.5, 0.05  # change these!
_, _, rewards_2, _ = run_q_learning(bandit, 1000, alpha_2, epsilon_2)

# Combination 3
alpha_3, epsilon_3 = 0.01, 0.3  # change these!
_, _, rewards_3, _ = run_q_learning(bandit, 1000, alpha_3, epsilon_3)

# Add more combinations!

print(f"Combo 1 (alpha={alpha_1}, eps={epsilon_1}): Total reward = {sum(rewards_1)}")
print(f"Combo 2 (alpha={alpha_2}, eps={epsilon_2}): Total reward = {sum(rewards_2)}")
print(f"Combo 3 (alpha={alpha_3}, eps={epsilon_3}): Total reward = {sum(rewards_3)}")

# What's the theoretical maximum? (choosing the best arm every time)
print(f"\nTheoretical max (always best arm): {int(1000 * max(bandit))}")
print(f"Expected with random: {int(1000 * np.mean(bandit))}")

---

## **Part 7: Bonus Challenges**

### Bonus 1: Changing Reward Probabilities

What happens if the bandit's reward probabilities are closer together (e.g., 0.4 vs 0.6)? Or further apart (e.g., 0.1 vs 0.9)? Try it and see how your agent performs!

Does the agent need different parameters when the problem is harder (probabilities closer together)?

### Bonus 2: Decaying Epsilon

In many real RL applications, the exploration rate decreases over time. The idea: explore a lot at the beginning, then exploit more as you learn.

Implement a modified version of `run_q_learning` where epsilon decays by a small amount each trial:

```python
epsilon = epsilon * decay_rate  # e.g., decay_rate = 0.999
```

Does this improve performance?

In [ ]:
pass

### Bonus 3: Connection to Neuroscience

The Q-learning update rule has a fascinating parallel in the brain. The **prediction error** term `(reward - Q[arm])` closely mirrors the firing patterns of **dopamine neurons** in the midbrain.

Research by Wolfram Schultz and colleagues showed that:
- When an unexpected reward occurs (positive prediction error), dopamine neurons fire
- When an expected reward is received, dopamine neurons show no change
- When an expected reward is absent (negative prediction error), dopamine neuron firing decreases

This connection between a simple computational algorithm and actual neural activity is one of the most celebrated findings in computational neuroscience!

The two-armed bandit task (and its multi-armed extensions) is widely used in cognitive neuroscience research to study how humans and animals learn from reward feedback.